# Qwen3-4B IT Ticket Classifier
## Fine-tuning with LoRA + Classification Head

### Fixes applied from previous experiments:
- Last token pooling (not mean pooling) — correct for decoder models
- Stratified stages — balanced department distribution per stage
- save_best_model per stage — saves best epoch not last
- NUM_EPOCHS=2 — prevents overfitting
- LoRA on ALL 7 layers (not just q+v)
- LoRA HPT: 3 configs compared automatically
- Full metrics saved: accuracy, F1, confusion matrix, per-class report

### Architecture (based on Yousefiramandi & Cooney 2025):
```
Qwen3-4B (frozen, 4-bit via Unsloth)
    + LoRA Adapters (trainable) — ALL 7 attention/MLP projections
    + Classification Head: Linear(2560 → 256 → 10)
    + Last Token Pooling
    + Cross-Entropy Loss
```

In [ ]:
import json
from pathlib import Path
from google.colab import drive, files

drive.mount('/content/drive')

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if "qwen3_4b_finetuning" in file.lower() and file.endswith(".ipynb"):
            print(os.path.join(root, file))

In [ ]:
import json
from pathlib import Path
from google.colab import drive, files


old_path = Path("/content/drive/MyDrive/Colab Notebooks/qwen3_4b_finetuning.ipynb")
new_path = Path("/content/drive/MyDrive/Colab Notebooks/qwen3_4b_finetuning.ipynb")

nb = json.loads(old_path.read_text(encoding="utf-8"))

# امسح widgets من metadata الرئيسية
nb.get("metadata", {}).pop("widgets", None)

# امسح outputs كمان احتياطيًا
for cell in nb.get("cells", []):
    cell["outputs"] = []
    cell["execution_count"] = None

new_path.write_text(json.dumps(nb, indent=1, ensure_ascii=False), encoding="utf-8")

# تحقق
check = json.loads(new_path.read_text(encoding="utf-8"))
print("widgets exists?", "widgets" in check.get("metadata", {}))
print("Saved fixed file:", new_path)

files.download(str(new_path))

## Cell 1 — Install Libraries

In [ ]:
# Run once per session
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q "transformers>=4.45.0,<5.0.0"
!pip install -q datasets scikit-learn matplotlib seaborn
print('✅ Installation done')

## Cell 2 — Imports

In [ ]:
import os, gc, json, torch, torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)
from torch.utils.data import Dataset, DataLoader
from transformers import get_cosine_schedule_with_warmup
from unsloth import FastLanguageModel
from peft import PeftModel

print('✅ All imports done')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

## Cell 3 — Configuration

خليني أحلل الداتا دلوقتي عشان أجاوبك بدقة.## الإجابة المباشرة

**ارجع لـ `MAX_SEQ_LEN = 256` — هو الأنسب لداتاتك.**

الأرقام بتقول:

- الـ **وسيط** هو **73 token** فقط — يعني نص الـ tickets قصيرة
- الـ 256 بيغطي **99.4%** من الـ tickets
- الـ 512 بيغطي **100%** لكن بيزود على الـ 256 بـ **165 ticket فقط** من أصل 29,650

المقايضة مش مناسبة خالص — بتضاعف وقت التدريب وخطر الـ OOM على Colab T4 عشان تغطي 0.6% إضافية.

**الـ 256 هو الـ sweet spot لداتاتك تحديداً.**

In [ ]:
# ════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════
DATA_PATH       = '/content/drive/MyDrive/IT Support Ticket Data.csv'
CHECKPOINT_BASE = '/content/drive/MyDrive/qwen3-cls-checkpoints'

# ════════════════════════════════════════════
# MODEL — Qwen3-4B (NOT Instruct version)
# Research showed Instruct version performs worse with classification head
# ════════════════════════════════════════════
MODEL_NAME = 'unsloth/Qwen3-4B-Instruct-2507-bnb-4bit'
MAX_SEQ_LEN = 256

# ════════════════════════════════════════════
# DATA SPLIT — identical to Classical ML
# ════════════════════════════════════════════
RANDOM_STATE = 42
TEST_SIZE    = 0.2
VAL_SIZE     = 0.5

# ════════════════════════════════════════════
# STAGED TRAINING
# ════════════════════════════════════════════
STAGE_SIZE = 3000
NUM_STAGES = 7

# ════════════════════════════════════════════
# TRAINING HYPERPARAMETERS
# FIX: NUM_EPOCHS=2 (was 3) — prevents overfitting
# ════════════════════════════════════════════
BATCH_SIZE   = 2
GRAD_ACCUM   = 16    # effective batch = 32
NUM_EPOCHS   = 2     # ← FIX: was 3, overfitting at epoch 3
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

# ════════════════════════════════════════════
# LORA — ALL 7 layers (not just q+v)
# Research: applying LoRA to all layers improves performance
# ════════════════════════════════════════════
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj'
]

# ════════════════════════════════════════════
# LORA HPT — 3 configs to compare
# Based on Yousefiramandi & Cooney (2025): r=8,16 both work well
# alpha = r (not 2r) — recommended by Unsloth docs
# ════════════════════════════════════════════
LORA_CONFIGS = [
    {'name': 'LoRA_r8',  'r': 8,  'lora_alpha': 8,  'lr': 5e-5, 'lora_dropout': 0.05},
    {'name': 'LoRA_r16', 'r': 16, 'lora_alpha': 16, 'lr': 1e-4, 'lora_dropout': 0.05},  # ← كان 2e-4
    {'name': 'LoRA_r32', 'r': 32, 'lora_alpha': 32, 'lr': 1e-4, 'lora_dropout': 0.05},
]

# ════════════════════════════════════════════
# CLASSES
# ════════════════════════════════════════════
DEPARTMENTS = [
    'Billing and Payments', 'Customer Service', 'General Inquiry',
    'Human Resources', 'IT Support', 'Product Support',
    'Returns and Exchanges', 'Sales and Pre-Sales',
    'Service Outages and Maintenance', 'Technical Support'
]
LABEL2ID = {d: i for i, d in enumerate(DEPARTMENTS)}
ID2LABEL = {i: d for i, d in enumerate(DEPARTMENTS)}
NUM_CLASSES = len(DEPARTMENTS)

print('✅ Config ready')
print(f'   Model       : {MODEL_NAME}')
print(f'   LoRA layers : {len(LORA_TARGET_MODULES)} (all projections)')
print(f'   Epochs      : {NUM_EPOCHS} per stage (overfitting fix)')
print(f'   HPT configs : {len(LORA_CONFIGS)}')

## Cell 4 — Mount Drive & Load Data

In [ ]:
drive.mount('/content/drive')
os.makedirs(CHECKPOINT_BASE, exist_ok=True)

df = pd.read_csv(DATA_PATH, index_col=0)
df = df.dropna(subset=['Body']).reset_index(drop=True)
df = df[['Body', 'Department']]

print(f'Total samples : {len(df)}')
print(f'Departments   : {df["Department"].nunique()}')
print()
print(df['Department'].value_counts())

## Cell 5 — Data Split + Stratified Stages

In [ ]:
# Same split as Classical ML
train_df, temp_df = train_test_split(
    df, test_size=TEST_SIZE,
    stratify=df['Department'], random_state=RANDOM_STATE
)
val_df, test_df = train_test_split(
    temp_df, test_size=VAL_SIZE,
    stratify=temp_df['Department'], random_state=RANDOM_STATE
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# ════════════════════════════════════════
# FIX: STRATIFIED stages (not sequential)
# Each stage has balanced department distribution
# ════════════════════════════════════════
def make_stratified_stages(df, stage_size, num_stages, random_state=42):
    """
    Creates non-overlapping stratified stages.
    Each stage has proportional representation of all departments.
    """
    total_needed = stage_size * num_stages
    assert len(df) >= total_needed, f'Need {total_needed} rows, have {len(df)}'

    # Sample total_needed rows with stratification
    sampled, _ = train_test_split(
        df, train_size=total_needed,
        stratify=df['Department'], random_state=random_state
    )
    sampled = sampled.reset_index(drop=True)

    # Split into num_stages equal stratified chunks
    stages = []
    remaining = sampled.copy()
    for i in range(num_stages - 1):
        stage, remaining = train_test_split(
            remaining, train_size=stage_size,
            stratify=remaining['Department'], random_state=random_state + i
        )
        stages.append(stage.reset_index(drop=True))
    stages.append(remaining.reset_index(drop=True))
    return stages

stages = make_stratified_stages(train_df, STAGE_SIZE, NUM_STAGES)

print('=' * 60)
print('  DATA SPLIT SUMMARY')
print('=' * 60)
print(f'  Train : {len(train_df):>6}  (80%)')
print(f'  Val   : {len(val_df):>6}  (10%)')
print(f'  Test  : {len(test_df):>6}  (10%)')
print(f'  Stages: {NUM_STAGES} × {STAGE_SIZE} = {NUM_STAGES*STAGE_SIZE} (stratified)')
print()
for i, s in enumerate(stages):
    counts = s['Department'].value_counts()
    print(f'  Stage {i+1}: min={counts.min()} max={counts.max()} '
          f'(balanced: {counts.std():.1f} std)')

## Cell 6 — Dataset Class & Tokenizer

In [ ]:
# Load tokenizer only (model loaded fresh per LoRA config)
_, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'


class TicketDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.texts     = dataframe['Body'].astype(str).tolist()
        self.labels    = [LABEL2ID[d] for d in dataframe['Department'].tolist()]
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }


val_loader  = DataLoader(TicketDataset(val_df,  tokenizer, MAX_SEQ_LEN), batch_size=BATCH_SIZE*2, shuffle=False)
test_loader = DataLoader(TicketDataset(test_df, tokenizer, MAX_SEQ_LEN), batch_size=BATCH_SIZE*2, shuffle=False)

print(f'✅ Tokenizer ready')
print(f'   Val  batches : {len(val_loader)}')
print(f'   Test batches : {len(test_loader)}')

## Cell 7 — Model: Qwen3-4B + LoRA + Classification Head

In [ ]:
class Qwen3Classifier(nn.Module):
    """
    Qwen3-4B (frozen, 4-bit via Unsloth)
        + LoRA Adapters (trainable)
        + Classification Head: Linear(hidden → 256 → num_classes)
        + LAST TOKEN POOLING (correct for decoder models)

    Based on Yousefiramandi & Cooney (2025):
    'the hidden state of the final token serves as a holistic
     representation of the entire sequence'
    """

    def __init__(self, model_name, num_classes, lora_cfg):
        super().__init__()

        # Step 1: Load Qwen3-4B base
        base_model, _ = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=MAX_SEQ_LEN,
            dtype=None,
            load_in_4bit=True,
        )

        # Step 2: Add LoRA on ALL 7 projection layers
        self.backbone = FastLanguageModel.get_peft_model(
            base_model,
            r=lora_cfg['r'],
            lora_alpha=lora_cfg['lora_alpha'],
            target_modules=LORA_TARGET_MODULES,
            lora_dropout=lora_cfg['lora_dropout'],
            bias='none',
            use_gradient_checkpointing='unsloth',
            random_state=RANDOM_STATE,
        )
        self.backbone.print_trainable_parameters()

        # Step 3: Classification Head — float32 دايماً للاستقرار
        hidden_size = self.backbone.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        ).to(device=self.backbone.device)  # float32 by default — stable loss

        # Xavier init
        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

        self.loss_fn = nn.CrossEntropyLoss()
        print(f'\n✅ Qwen3Classifier ready')
        print(f'   Hidden size     : {hidden_size}')
        print(f'   Head dtype      : {next(self.classifier.parameters()).dtype}')
        print(f'   Head            : {hidden_size} → 256 → {num_classes}')
        print(f'   Pooling         : last token (decoder-correct)')

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        # LAST TOKEN POOLING — correct for decoder models
        last_hidden = outputs.hidden_states[-1]  # (B, L, hidden)

        # Find last non-padding token for each sample
        seq_lengths = attention_mask.sum(dim=1) - 1  # (B,)
        batch_size  = input_ids.shape[0]
        pooled = last_hidden[
            torch.arange(batch_size, device=input_ids.device),
            seq_lengths
        ]  # (B, hidden16)

        # Cast to float32 — backbone is float16, classifier is float32
        pooled = pooled.float()

        logits = self.classifier(pooled)  # (B, num_classes) — float32

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {'loss': loss, 'logits': logits}


def build_model(lora_cfg):
    """Build fresh model for each LoRA config"""
    gc.collect()
    torch.cuda.empty_cache()
    model = Qwen3Classifier(MODEL_NAME, NUM_CLASSES, lora_cfg)
    return model


print('✅ Model class defined')

## Cell 8 — Evaluation Function (Full Metrics)

In [ ]:
def evaluate(model, loader, split_name='Val', show_plots=True, save_dir=None):
    """
    Full evaluation: accuracy, F1 (macro/weighted), per-class report,
    confusion matrix plot, and optionally saves everything.
    """
    model.eval()
    device     = next(model.parameters()).device
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels']
            out            = model(input_ids=input_ids, attention_mask=attention_mask)
            preds          = out['logits'].argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc  = accuracy_score(all_labels, all_preds)
    f1m  = f1_score(all_labels, all_preds, average='macro',    zero_division=0)
    f1w  = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)
    cm   = confusion_matrix(all_labels, all_preds)
    report = classification_report(
        all_labels, all_preds,
        target_names=DEPARTMENTS, zero_division=0, output_dict=True
    )

    # ── Print
    print('=' * 65)
    print(f'  {split_name}')
    print('=' * 65)
    print(f'  Accuracy    : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  F1 Macro    : {f1m:.4f}')
    print(f'  F1 Weighted : {f1w:.4f}')
    print()
    print(classification_report(
        all_labels, all_preds,
        target_names=DEPARTMENTS, zero_division=0
    ))

    # ── Plots
    if show_plots:
        fig = plt.figure(figsize=(18, 7))
        gs  = gridspec.GridSpec(1, 2, width_ratios=[1.8, 1])

        # Confusion matrix
        ax1 = fig.add_subplot(gs[0])
        short = [d[:18] for d in DEPARTMENTS]
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=short, yticklabels=short, ax=ax1)
        ax1.set_title(f'Confusion Matrix — {split_name}', fontsize=13)
        ax1.set_xlabel('Predicted'); ax1.set_ylabel('True')
        plt.setp(ax1.get_xticklabels(), rotation=45, ha='right', fontsize=8)
        plt.setp(ax1.get_yticklabels(), rotation=0, fontsize=8)

        # F1 per class
        ax2 = fig.add_subplot(gs[1])
        colors = ['#2ecc71' if f >= 0.6 else '#e74c3c' if f < 0.4 else '#f39c12'
                  for f in f1_per_class]
        ax2.barh(range(NUM_CLASSES), f1_per_class, color=colors)
        ax2.set_yticks(range(NUM_CLASSES))
        ax2.set_yticklabels([d[:20] for d in DEPARTMENTS], fontsize=8)
        ax2.set_xlabel('F1 Score')
        ax2.set_title('F1 per Department', fontsize=11)
        ax2.axvline(x=f1m, color='navy', linestyle='--', alpha=0.7,
                    label=f'Macro F1={f1m:.3f}')
        ax2.legend(fontsize=9)
        ax2.set_xlim(0, 1)

        plt.suptitle(f'{split_name} | Acc={acc:.3f} | F1-Macro={f1m:.3f}',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()

        if save_dir:
            plot_path = os.path.join(save_dir, f'eval_{split_name.replace(" ","_")}.png')
            plt.savefig(plot_path, dpi=100, bbox_inches='tight')
            print(f'  Plot saved → {plot_path}')
        plt.show()

    model.train()
    return {
        'accuracy': acc, 'f1_macro': f1m, 'f1_weighted': f1w,
        'f1_per_class': f1_per_class.tolist(),
        'confusion_matrix': cm.tolist(),
        'per_class_report': report
    }


print('✅ evaluate() defined')

## Cell 9 — Checkpoint Functions (Full State)

In [ ]:
def save_checkpoint(model, optimizer, scheduler, stage, epoch,
                    lora_name, metrics, is_best=False):
    """
    Save complete checkpoint:
    - LoRA adapters
    - Classification head
    - Optimizer + scheduler state
    - Full metrics (accuracy, F1, CM, per-class)
    - Best model flag
    """
    tag      = 'BEST' if is_best else f'stage{stage}_ep{epoch}'
    ckpt_dir = os.path.join(CHECKPOINT_BASE, lora_name, tag)
    os.makedirs(ckpt_dir, exist_ok=True)

    # 1. LoRA adapters
    model.backbone.save_pretrained(os.path.join(ckpt_dir, 'lora_adapters'))

    # 2. Classification head
    torch.save(model.classifier.state_dict(),
               os.path.join(ckpt_dir, 'classifier_head.pt'))

    # 3. Training state
    torch.save({
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler else None,
        'stage': stage, 'epoch': epoch, 'metrics': metrics
    }, os.path.join(ckpt_dir, 'training_state.pt'))

    # 4. Full metrics JSON (human readable)
    metrics_to_save = {
        'stage': stage, 'epoch': epoch,
        'lora_config': lora_name,
        'is_best': is_best,
        'accuracy': round(metrics['accuracy'], 6),
        'f1_macro': round(metrics['f1_macro'], 6),
        'f1_weighted': round(metrics['f1_weighted'], 6),
        'f1_per_class': {
            DEPARTMENTS[i]: round(v, 4)
            for i, v in enumerate(metrics['f1_per_class'])
        },
        'confusion_matrix': metrics['confusion_matrix'],
        'per_class_report': metrics['per_class_report']
    }
    with open(os.path.join(ckpt_dir, 'metrics.json'), 'w') as f:
        json.dump(metrics_to_save, f, indent=2)

    label = '🏆 BEST' if is_best else '💾'
    print(f'  {label} Checkpoint saved → {ckpt_dir}')
    print(f'     acc={metrics["accuracy"]:.4f}  '
          f'F1-macro={metrics["f1_macro"]:.4f}  '
          f'F1-weighted={metrics["f1_weighted"]:.4f}')


def load_checkpoint(model, optimizer, scheduler, stage, epoch, lora_name):
    tag      = f'stage{stage}_ep{epoch}'
    ckpt_dir = os.path.join(CHECKPOINT_BASE, lora_name, tag)
    if not os.path.exists(ckpt_dir):
        print(f'  ❌ Not found: {ckpt_dir}')
        return False

    model.backbone = PeftModel.from_pretrained(
        model.backbone, os.path.join(ckpt_dir, 'lora_adapters'),
        is_trainable=True
    )
    model.classifier.load_state_dict(
        torch.load(os.path.join(ckpt_dir, 'classifier_head.pt'),
                   map_location='cpu')
    )
    state = torch.load(os.path.join(ckpt_dir, 'training_state.pt'),
                       map_location='cpu')
    optimizer.load_state_dict(state['optimizer'])
    if scheduler and state['scheduler']:
        scheduler.load_state_dict(state['scheduler'])
    print(f'  ✅ Loaded from {ckpt_dir}')
    print(f'     acc={state["metrics"]["accuracy"]:.4f}  '
          f'F1-macro={state["metrics"]["f1_macro"]:.4f}')
    return True


def list_checkpoints(lora_name=None):
    """List all saved checkpoints with their metrics"""
    base = os.path.join(CHECKPOINT_BASE, lora_name) if lora_name else CHECKPOINT_BASE
    if not os.path.exists(base):
        print('No checkpoints found.')
        return
    print(f'Checkpoints in {base}:')
    for root, dirs, files in os.walk(base):
        if 'metrics.json' in files:
            with open(os.path.join(root, 'metrics.json')) as f:
                m = json.load(f)
            best_flag = ' ← BEST' if m.get('is_best') else ''
            print(f'  {os.path.relpath(root, CHECKPOINT_BASE):40s} '
                  f'acc={m["accuracy"]:.4f} F1={m["f1_macro"]:.4f}{best_flag}')


print('✅ Checkpoint functions defined')

## Cell 10 — Training Loop (One Stage with save_best_model)

In [ ]:
def train_one_stage(model, stage_df, optimizer, scheduler,
                    stage_num, lora_name, best_val_f1):
    """
    Train for NUM_EPOCHS on one stage.
    FIX: save_best_model — saves best epoch based on Val F1-Macro
    Returns: (stage_metrics, best_val_f1_updated)
    """
    device       = next(model.parameters()).device
    stage_ds     = TicketDataset(stage_df, tokenizer, MAX_SEQ_LEN)
    train_loader = DataLoader(stage_ds, batch_size=BATCH_SIZE, shuffle=True)

    print(f'\n{"="*65}')
    print(f'  STAGE {stage_num}/{NUM_STAGES}  |  {lora_name}')
    print(f'  Samples : {len(stage_df)} | Batches : {len(train_loader)} | Epochs : {NUM_EPOCHS}')
    print(f'{"="*65}')

    stage_best_f1      = best_val_f1
    stage_best_metrics = None

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        correct = total = 0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            out  = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out['loss'] / GRAD_ACCUM
            loss.backward()

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 0.3
                )
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            total_loss += out['loss'].item()
            preds = out['logits'].argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            if (step + 1) % 100 == 0:
                print(f'   Epoch {epoch} | Step {step+1:>4}/{len(train_loader)} | '
                      f'Loss: {total_loss/(step+1):.4f} | '
                      f'Train Acc: {correct/total:.4f}')

        train_acc = correct / total
        avg_loss  = total_loss / len(train_loader)
        print(f'\n  ── Epoch {epoch} done | Loss={avg_loss:.4f} | Train Acc={train_acc:.4f}')

        # ════════════════════════════════════════
        # FIX: Evaluate after EVERY epoch
        # Save checkpoint only if Val F1 improved
        # ════════════════════════════════════════
        ckpt_dir = os.path.join(CHECKPOINT_BASE, lora_name, f'stage{stage_num}_ep{epoch}')
        val_metrics = evaluate(
            model, val_loader,
            split_name=f'Val S{stage_num}/E{epoch}',
            show_plots=False
        )
        val_metrics['train_acc']  = train_acc
        val_metrics['train_loss'] = avg_loss

        # Always save per-epoch checkpoint
        save_checkpoint(
            model, optimizer, scheduler,
            stage_num, epoch, lora_name, val_metrics, is_best=False
        )

        # Save BEST model if Val F1 improved
        if val_metrics['f1_macro'] > stage_best_f1:
            stage_best_f1      = val_metrics['f1_macro']
            stage_best_metrics = val_metrics
            save_checkpoint(
                model, optimizer, scheduler,
                stage_num, epoch, lora_name, val_metrics, is_best=True
            )

        print(f'  Val Acc={val_metrics["accuracy"]:.4f} | '
              f'F1-Macro={val_metrics["f1_macro"]:.4f} | '
              f'Best F1 so far={stage_best_f1:.4f}')

    # Save confusion matrix plot for the stage
    if stage_best_metrics:
        stage_ckpt_dir = os.path.join(CHECKPOINT_BASE, lora_name, 'BEST')
        # Re-run eval with plots at end of stage
        full_metrics = evaluate(
            model, val_loader,
            split_name=f'Stage {stage_num} Final',
            show_plots=True,
            save_dir=os.path.join(CHECKPOINT_BASE, lora_name)
        )

    return stage_best_metrics or val_metrics, stage_best_f1


print('✅ train_one_stage() defined')

In [ ]:
# ════════════════════════════════════════════
# EMERGENCY FIX — Run this before Cell 11
# Patches the model in-place without rebuild
# ════════════════════════════════════════════

import torch.nn as nn

# 1. Convert classifier to float32 for stable loss
for layer in model.classifier:
    if isinstance(layer, nn.Linear):
        layer.weight.data = layer.weight.data.float()
        layer.bias.data   = layer.bias.data.float()

# 2. Override forward to fix dtype + loss
def fixed_forward(self, input_ids, attention_mask, labels=None):
    outputs = self.backbone(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True
    )
    last_hidden = outputs.hidden_states[-1]
    seq_lengths = attention_mask.sum(dim=1) - 1
    batch_size  = input_ids.shape[0]
    pooled = last_hidden[
        torch.arange(batch_size, device=input_ids.device),
        seq_lengths
    ]
    # Cast to float32 before classifier
    pooled = pooled.float()
    logits = self.classifier(pooled)

    loss = None
    if labels is not None:
        loss = self.loss_fn(logits, labels)

    return {'loss': loss, 'logits': logits}

import types
model.forward = types.MethodType(fixed_forward, model)

# 3. Reduce clip_grad_norm
# (already set in train_one_stage — make sure it's 0.3)

print('✅ Fix applied — classifier is float32, pooled cast to float32')
print(f'   Classifier dtype: {next(model.classifier.parameters()).dtype}')

## Cell 11 — Main Training Loop (LoRA HPT)

استهلاك gpu and system ram أٌقل

In [ ]:
# ════════════════════════════════════════
# SELECT WHICH LORA CONFIG TO RUN
# 0 = LoRA_r8  (conservative, fast)
# 1 = LoRA_r16 (balanced) ← start here
# 2 = LoRA_r32 (aggressive)
# ════════════════════════════════════════
ACTIVE_CONFIG_IDX = 1  # ← change to 0 or 2 for other configs

gc.collect()
torch.cuda.empty_cache()

cfg       = LORA_CONFIGS[ACTIVE_CONFIG_IDX]
lora_name = cfg['name']

print(f'Starting training: {lora_name}')
print(f'  r={cfg["r"]}  alpha={cfg["lora_alpha"]}  '
      f'lr={cfg["lr"]}  dropout={cfg["lora_dropout"]}')
print(f'  Target modules: {len(LORA_TARGET_MODULES)} layers')
print(f'  Epochs per stage: {NUM_EPOCHS}')

# Build model
model     = build_model(cfg)
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=cfg['lr'],
                               weight_decay=WEIGHT_DECAY)

# Cosine schedule over ALL stages combined
steps_per_stage = (STAGE_SIZE // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
total_steps     = steps_per_stage * NUM_STAGES
warmup_steps    = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'  Total steps : {total_steps}')
print(f'  Warmup steps: {warmup_steps}')

# ── Run all stages
all_stage_metrics = []
best_val_f1 = 0.0

for stage_num in range(1, NUM_STAGES + 1):
    stage_metrics, best_val_f1 = train_one_stage(
        model, stages[stage_num - 1],
        optimizer, scheduler,
        stage_num, lora_name, best_val_f1
    )
    all_stage_metrics.append({'stage': stage_num, **stage_metrics})

# ── Save full training summary
summary_path = os.path.join(CHECKPOINT_BASE, lora_name, 'training_summary.json')
with open(summary_path, 'w') as f:
    json.dump(all_stage_metrics, f, indent=2, default=str)

print(f'\n{"="*65}')
print(f'  TRAINING COMPLETE — {lora_name}')
print(f'  Best Val F1-Macro : {best_val_f1:.4f}')
print(f'  Summary saved     : {summary_path}')
print(f'{"="*65}')

## Cell 12 — Resume from Checkpoint

In [ ]:
# ════════════════════════════════════════
# Use this when Colab disconnects
# Run Cells 1→10 first, then this cell
# ════════════════════════════════════════
RESUME_LORA_NAME  = 'LoRA_r16'  # which config
RESUME_FROM_STAGE = 3           # last COMPLETED stage
RESUME_FROM_EPOCH = 2           # last completed epoch in that stage

gc.collect(); torch.cuda.empty_cache()

cfg       = next(c for c in LORA_CONFIGS if c['name'] == RESUME_LORA_NAME)
lora_name = cfg['name']
model     = build_model(cfg)
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=cfg['lr'], weight_decay=WEIGHT_DECAY)

steps_per_stage = (STAGE_SIZE // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
total_steps     = steps_per_stage * NUM_STAGES
warmup_steps    = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

success = load_checkpoint(
    model, optimizer, scheduler,
    RESUME_FROM_STAGE, RESUME_FROM_EPOCH, lora_name
)

if success:
    # Get best F1 so far from saved checkpoints
    best_val_f1 = 0.0
    best_ckpt = os.path.join(CHECKPOINT_BASE, lora_name, 'BEST', 'metrics.json')
    if os.path.exists(best_ckpt):
        with open(best_ckpt) as f:
            best_val_f1 = json.load(f)['f1_macro']
        print(f'  Best F1 so far: {best_val_f1:.4f}')

    print(f'\n✅ Resuming {lora_name} from Stage {RESUME_FROM_STAGE}')
    all_stage_metrics = []
    for stage_num in range(RESUME_FROM_STAGE + 1, NUM_STAGES + 1):
        stage_metrics, best_val_f1 = train_one_stage(
            model, stages[stage_num - 1],
            optimizer, scheduler,
            stage_num, lora_name, best_val_f1
        )
        all_stage_metrics.append({'stage': stage_num, **stage_metrics})

## Cell 13 — Final Test Evaluation

In [ ]:
# Load best checkpoint before test evaluation
best_ckpt_dir = os.path.join(CHECKPOINT_BASE, lora_name, 'BEST')
if os.path.exists(best_ckpt_dir):
    model.classifier.load_state_dict(
        torch.load(os.path.join(best_ckpt_dir, 'classifier_head.pt'),
                   map_location='cpu')
    )
    print(f'✅ Loaded BEST model from {best_ckpt_dir}')

print(f'Final test evaluation — {lora_name}')
print(f'Test set size: {len(test_df)} (same split as Classical ML)\n')

test_dir = os.path.join(CHECKPOINT_BASE, lora_name, 'final_test')
os.makedirs(test_dir, exist_ok=True)

test_metrics = evaluate(
    model, test_loader,
    split_name=f'TEST — {lora_name}',
    show_plots=True,
    save_dir=test_dir
)

# Save test metrics
with open(os.path.join(test_dir, 'test_metrics.json'), 'w') as f:
    json.dump({'lora_config': lora_name, **test_metrics}, f, indent=2, default=str)

print(f'\n✅ Test metrics saved → {test_dir}')

## Cell 14 — LoRA HPT Comparison + Learning Curves

In [ ]:
# Run after completing all 3 LoRA configs

# ── 1. Collect results
rows = []
for cfg in LORA_CONFIGS:
    tf = os.path.join(CHECKPOINT_BASE, cfg['name'], 'final_test', 'test_metrics.json')
    if os.path.exists(tf):
        with open(tf) as f: m = json.load(f)
        rows.append({
            'Config': cfg['name'], 'r': cfg['r'],
            'alpha': cfg['lora_alpha'], 'lr': cfg['lr'],
            'Test Acc': round(m['accuracy'], 4),
            'F1 Macro': round(m['f1_macro'], 4),
            'F1 Weighted': round(m['f1_weighted'], 4)
        })
    else:
        rows.append({'Config': cfg['name'], 'r': cfg['r'],
                     'alpha': cfg['lora_alpha'], 'lr': cfg['lr'],
                     'Test Acc': 'N/A', 'F1 Macro': 'N/A', 'F1 Weighted': 'N/A'})

hpt_df = pd.DataFrame(rows)

print('=' * 72)
print('  LORA HPT RESULTS — Qwen3-4B')
print('=' * 72)
print(hpt_df.to_string(index=False))

# ── 2. Compare with Classical ML baseline
print('\n' + '=' * 72)
print('  COMPARISON WITH CLASSICAL ML')
print('=' * 72)
classical = pd.DataFrame([
    {'Model': 'Naive Bayes (BoW)',                'Accuracy': 0.4415, 'F1 Macro': 0.40},
    {'Model': 'Logistic Regression (TF-IDF)',     'Accuracy': 0.5089, 'F1 Macro': 0.51},
    {'Model': 'Logistic Regression (GridSearch)', 'Accuracy': 0.6378, 'F1 Macro': 0.64},
    {'Model': 'Random Forest (Word2Vec)',         'Accuracy': 0.6809, 'F1 Macro': 0.68},
])
print(classical.to_string(index=False))

# ── 3. Learning curves per config
fig, axes = plt.subplots(1, len(LORA_CONFIGS), figsize=(6*len(LORA_CONFIGS), 5))
if len(LORA_CONFIGS) == 1: axes = [axes]

for ax, cfg in zip(axes, LORA_CONFIGS):
    summary_path = os.path.join(CHECKPOINT_BASE, cfg['name'], 'training_summary.json')
    if not os.path.exists(summary_path):
        ax.set_title(f"{cfg['name']} — no data")
        continue
    with open(summary_path) as f:
        summary = json.load(f)

    stages_x = [s['stage'] for s in summary]
    accs     = [s['accuracy'] for s in summary]
    f1s      = [s['f1_macro'] for s in summary]

    ax.plot(stages_x, accs, 'b-o', label='Val Accuracy', linewidth=2)
    ax.plot(stages_x, f1s,  'r-s', label='Val F1 Macro', linewidth=2)
    ax.axhline(y=0.681, color='green', linestyle='--', alpha=0.7, label='RF baseline (68.1%)')
    ax.set_xlabel('Stage')
    ax.set_ylabel('Score')
    ax.set_title(f"{cfg['name']} (r={cfg['r']}, lr={cfg['lr']})")
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)

plt.suptitle('Qwen3-4B — LoRA HPT Learning Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
comparison_path = os.path.join(CHECKPOINT_BASE, 'HPT_comparison.png')
plt.savefig(comparison_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'\n✅ HPT comparison saved → {comparison_path}')

## Cell 15 — List All Checkpoints

In [ ]:
list_checkpoints()